# Lead Source Quality Analysis

Analyze CRM lead-generation data to compare lead quality, conversion rate, sales cycle time, and eventual value across lead sources. The goal is to identify which sources generate volume, which generate quality, and which actually produce revenue efficiently.


## What This Notebook Answers

- Which lead sources generate the most leads?
- Which sources bring in the highest-quality leads?
- Which sources convert best from lead to converted lead and from lead to closed-won deal?
- Which sources close fastest?
- Which sources deliver the highest eventual deal value?
- What do these patterns imply for marketing and sales strategy?


## Dataset And Metric Definitions

**Dataset:** public CRM dashboard repository with four CSV files:

- `leads.csv`: source, created date, status, industry, lead score
- `deals.csv`: deal size, stage, close date
- `activities.csv`: downstream sales activity records
- `sources.csv`: source lookup table

Definitions used here:

- **Lead quality:** average `lead_score`, plus rates of `Qualified`, `Converted`, and `Closed Won`
- **Lead conversion rate:** share of leads whose `status` is `Converted`
- **Win rate:** share of leads that reached a `Closed Won` deal stage
- **Sales cycle time:** days from `created_date` to `close_date` for leads with deals
- **Eventual value:** average and total `deal_size`, especially for `Closed Won` deals

Guardrail:

- This is CRM-style snapshot data, not full marketing attribution data.
- Lead source should therefore be interpreted as the recorded origin source for the lead, not a multi-touch acquisition history.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

BASE_URL = "https://raw.githubusercontent.com/abhay1967/CRM-DASHBOARD/main"
URLS = {
    "leads": f"{BASE_URL}/leads.csv",
    "deals": f"{BASE_URL}/deals.csv",
    "activities": f"{BASE_URL}/activities.csv",
    "sources": f"{BASE_URL}/sources.csv",
}

print(URLS)


{'leads': 'https://raw.githubusercontent.com/abhay1967/CRM-DASHBOARD/main/leads.csv', 'deals': 'https://raw.githubusercontent.com/abhay1967/CRM-DASHBOARD/main/deals.csv', 'activities': 'https://raw.githubusercontent.com/abhay1967/CRM-DASHBOARD/main/activities.csv', 'sources': 'https://raw.githubusercontent.com/abhay1967/CRM-DASHBOARD/main/sources.csv'}


## Load The CRM Tables

Load the raw source tables and inspect their sizes before building the merged lead-quality model.


In [2]:
leads = pd.read_csv(URLS["leads"])
deals = pd.read_csv(URLS["deals"])
activities = pd.read_csv(URLS["activities"])
sources = pd.read_csv(URLS["sources"])

shape_frame = pd.DataFrame(
    {
        "table": ["leads", "deals", "activities", "sources"],
        "rows": [len(leads), len(deals), len(activities), len(sources)],
        "columns": [leads.shape[1], deals.shape[1], activities.shape[1], sources.shape[1]],
    }
)

display(shape_frame)
display(leads.head())
display(deals.head())
display(activities.head())


,table,rows,columns
0,leads,500,7
1,deals,350,5
2,activities,600,4
3,sources,8,2


,lead_id,lead_name,industry,source,created_date,status,lead_score
0,L0001,HDFC Analytics 1,Manufacturing,Email,2024-03-29,Lost,87
1,L0002,Udaan Pvt Ltd 2,Manufacturing,Cold Call,2024-11-19,Qualified,70
2,L0003,Udaan Pvt Ltd 3,Healthcare,LinkedIn Ads,2024-12-24,Converted,79
3,L0004,Infosys Ltd 4,Retail,Referral,2024-10-08,Proposal,27
4,L0005,Reliance Digital 5,Manufacturing,Cold Call,2024-12-04,Converted,87


,deal_id,lead_id,deal_size,stage,close_date
0,D0001,L0211,349497,Closed Won,2024-06-26
1,D0002,L0324,444064,Negotiation,2024-10-16
2,D0003,L0252,284858,Closed Won,2024-09-04
3,D0004,L0070,97393,Negotiation,2024-07-06
4,D0005,L0002,298518,Closed Won,2025-02-03


,activity_id,lead_id,activity_type,activity_date
0,A0001,L0463,Call,2024-01-02
1,A0002,L0391,Email,2024-07-04
2,A0003,L0178,Meeting,2024-12-15
3,A0004,L0180,Meeting,2024-05-22
4,A0005,L0202,Email,2024-01-06


## Validation And Merge

Parse dates, standardize categories, and join deals and activities back to the lead table. The merged table becomes the analysis surface for source quality.


In [3]:
validation_report = pd.DataFrame(
    {
        "metric": [
            "lead rows",
            "deal rows",
            "activity rows",
            "missing lead source",
            "missing created dates",
            "missing lead scores",
            "deals without matching lead",
        ],
        "value": [
            len(leads),
            len(deals),
            len(activities),
            int(leads["source"].isna().sum()),
            int(leads["created_date"].isna().sum()),
            int(leads["lead_score"].isna().sum()),
            int((~deals["lead_id"].isin(leads["lead_id"])).sum()),
        ],
    }
)

leads["created_date"] = pd.to_datetime(leads["created_date"])
deals["close_date"] = pd.to_datetime(deals["close_date"])
activities["activity_date"] = pd.to_datetime(activities["activity_date"])

deal_summary = (
    deals.sort_values(["lead_id", "close_date"])
    .groupby("lead_id", as_index=False)
    .agg(
        max_deal_size=("deal_size", "max"),
        total_deal_value=("deal_size", "sum"),
        latest_stage=("stage", "last"),
        close_date=("close_date", "max"),
        deal_count=("deal_id", "count"),
    )
)

activity_summary = (
    activities.groupby("lead_id", as_index=False)
    .agg(
        activity_count=("activity_id", "count"),
        first_activity_date=("activity_date", "min"),
        last_activity_date=("activity_date", "max"),
    )
)

df = leads.merge(deal_summary, on="lead_id", how="left").merge(activity_summary, on="lead_id", how="left")
df["is_qualified"] = df["status"].eq("Qualified")
df["is_converted"] = df["status"].eq("Converted")
df["is_closed_won"] = df["latest_stage"].eq("Closed Won")
df["is_closed_lost"] = df["latest_stage"].eq("Closed Lost")
df["sales_cycle_days"] = (df["close_date"] - df["created_date"]).dt.days
df["lead_age_days"] = (pd.Timestamp("2024-12-31") - df["created_date"]).dt.days

display(validation_report)
display(df.head())


,metric,value
0,lead rows,500
1,deal rows,350
2,activity rows,600
3,missing lead source,0
4,missing created dates,0
5,missing lead scores,0
6,deals without matching lead,0


,lead_id,lead_name,industry,source,created_date,status,lead_score,max_deal_size,total_deal_value,latest_stage,close_date,deal_count,activity_count,first_activity_date,last_activity_date,is_qualified,is_converted,is_closed_won,is_closed_lost,sales_cycle_days,lead_age_days
0,L0001,HDFC Analytics 1,Manufacturing,Email,2024-03-29,Lost,87,"434,598.0000","434,598.0000",Negotiation,2024-05-15,1.0000,2.0000,2024-01-08,2024-09-27,False,False,False,False,47.0000,277
1,L0002,Udaan Pvt Ltd 2,Manufacturing,Cold Call,2024-11-19,Qualified,70,"298,518.0000","298,518.0000",Closed Won,2025-02-03,1.0000,NaN,NaT,NaT,True,False,True,False,76.0000,42
2,L0003,Udaan Pvt Ltd 3,Healthcare,LinkedIn Ads,2024-12-24,Converted,79,NaN,NaN,NaN,NaT,NaN,3.0000,2024-08-31,2024-10-29,False,True,False,False,NaN,7
3,L0004,Infosys Ltd 4,Retail,Referral,2024-10-08,Proposal,27,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaT,False,False,False,False,NaN,84
4,L0005,Reliance Digital 5,Manufacturing,Cold Call,2024-12-04,Converted,87,"100,311.0000","100,311.0000",Closed Won,2025-02-14,1.0000,2.0000,2024-02-10,2024-05-12,False,True,True,False,72.0000,27


## Overall Lead Funnel Snapshot

Establish the full funnel context before splitting performance by source.


In [4]:
overall = pd.DataFrame(
    {
        "metric": [
            "Leads",
            "Qualified leads",
            "Converted leads",
            "Closed won leads",
            "Closed lost leads",
            "Average lead score",
            "Average deal size",
            "Average sales cycle days",
        ],
        "value": [
            len(df),
            df["is_qualified"].sum(),
            df["is_converted"].sum(),
            df["is_closed_won"].sum(),
            df["is_closed_lost"].sum(),
            df["lead_score"].mean(),
            df.loc[df["max_deal_size"].notna(), "max_deal_size"].mean(),
            df.loc[df["sales_cycle_days"].notna(), "sales_cycle_days"].mean(),
        ],
    }
)

display(overall)


,metric,value
0,Leads,500.0000
1,Qualified leads,109.0000
2,Converted leads,103.0000
3,Closed won leads,109.0000
4,Closed lost leads,0.0000
5,Average lead score,61.9180
6,Average deal size,"282,094.5686"
7,Average sales cycle days,61.9686


## Lead Source Quality Table

Aggregate the merged CRM data by lead source. This becomes the primary decision table for source quality.


In [5]:
source_summary = (
    df.groupby("source")
    .agg(
        leads=("lead_id", "count"),
        avg_lead_score=("lead_score", "mean"),
        qualified_rate=("is_qualified", "mean"),
        conversion_rate=("is_converted", "mean"),
        win_rate=("is_closed_won", "mean"),
        avg_sales_cycle_days=("sales_cycle_days", "mean"),
        median_sales_cycle_days=("sales_cycle_days", "median"),
        avg_deal_size=("max_deal_size", "mean"),
        total_pipeline_value=("total_deal_value", "sum"),
        won_value=("max_deal_size", lambda s: s[df.loc[s.index, 'is_closed_won']].sum()),
        avg_activity_count=("activity_count", "mean"),
    )
    .reset_index()
)

source_summary["lead_share"] = source_summary["leads"] / source_summary["leads"].sum()
source_summary["won_value_share"] = source_summary["won_value"] / source_summary["won_value"].sum()
source_summary["quality_index"] = 100 * (
    0.30 * source_summary["avg_lead_score"].rank(pct=True)
    + 0.25 * source_summary["conversion_rate"].rank(pct=True)
    + 0.20 * source_summary["win_rate"].rank(pct=True)
    + 0.15 * source_summary["avg_deal_size"].rank(pct=True)
    + 0.10 * (1 - source_summary["avg_sales_cycle_days"].rank(pct=True))
)

display(source_summary.sort_values("quality_index", ascending=False).round(4))


,source,leads,avg_lead_score,qualified_rate,conversion_rate,win_rate,avg_sales_cycle_days,median_sales_cycle_days,avg_deal_size,total_pipeline_value,won_value,avg_activity_count,lead_share,won_value_share,quality_index
0,Cold Call,58,70.3448,0.2069,0.1897,0.2586,60.2195,65.0000,"303,676.7805","12,450,748.0000","4,088,799.0000",1.7692,0.1160,0.1351,78.7500
4,LinkedIn Ads,51,64.9804,0.1373,0.3725,0.1961,62.5000,60.0000,"293,380.0556","10,561,682.0000","2,391,849.0000",1.9062,0.1020,0.0790,77.5000
3,Google Ads,61,61.8689,0.2131,0.2131,0.2787,56.0930,56.0000,"283,076.3721","12,172,284.0000","4,856,065.0000",2.0000,0.1220,0.1605,68.1250
1,Email,72,60.0417,0.2639,0.1667,0.2917,57.2245,62.0000,"297,351.8980","14,570,243.0000","5,976,606.0000",1.6000,0.1440,0.1975,61.2500
5,Referral,63,61.3492,0.2063,0.2381,0.2063,67.3158,69.5000,"278,318.8684","10,576,117.0000","3,335,274.0000",1.4444,0.1260,0.1102,56.2500
7,Website,65,62.2000,0.2154,0.1538,0.1692,67.4186,61.0000,"272,491.1860","11,717,121.0000","3,056,468.0000",1.7447,0.1300,0.1010,37.5000
6,Social Media,71,56.9014,0.2254,0.2254,0.1549,62.8491,55.0000,"247,958.9434","13,141,824.0000","3,182,030.0000",1.6939,0.1420,0.1052,30.6250
2,Events,59,59.6610,0.2542,0.1186,0.1864,63.1064,59.0000,"288,150.6383","13,543,080.0000","3,371,594.0000",1.5789,0.1180,0.1114,30.0000


## Volume Vs Quality

Some lead sources generate lots of names but weak downstream outcomes. This section separates high-volume sources from high-quality sources.


In [6]:
plt.figure(figsize=(13, 8))
sns.scatterplot(
    data=source_summary,
    x="lead_share",
    y="conversion_rate",
    size="won_value",
    hue="avg_lead_score",
    sizes=(200, 1400),
    palette="viridis",
)
for _, row in source_summary.iterrows():
    plt.text(row["lead_share"], row["conversion_rate"], row["source"], fontsize=10)
plt.title("Lead Source Volume vs Conversion Rate")
plt.xlabel("Lead share")
plt.ylabel("Lead conversion rate")
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_37436\2172365760.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sales Cycle Time By Source

Faster sources can be strategically valuable even if they do not deliver the biggest deals. This view compares close speed and cycle spread across lead sources.


In [7]:
cycle_frame = df[df["sales_cycle_days"].notna()].copy()

display(
    source_summary[["source", "avg_sales_cycle_days", "median_sales_cycle_days", "conversion_rate", "win_rate", "avg_deal_size"]]
    .sort_values("avg_sales_cycle_days")
    .round(3)
)

plt.figure(figsize=(13, 6))
sns.boxplot(data=cycle_frame, x="source", y="sales_cycle_days")
plt.title("Sales Cycle Distribution By Lead Source")
plt.xlabel("")
plt.ylabel("Days from lead creation to close date")
plt.xticks(rotation=20)
plt.show()


,source,avg_sales_cycle_days,median_sales_cycle_days,conversion_rate,win_rate,avg_deal_size
3,Google Ads,56.0930,56.0000,0.2130,0.2790,"283,076.3720"
1,Email,57.2240,62.0000,0.1670,0.2920,"297,351.8980"
0,Cold Call,60.2200,65.0000,0.1900,0.2590,"303,676.7800"
4,LinkedIn Ads,62.5000,60.0000,0.3730,0.1960,"293,380.0560"
6,Social Media,62.8490,55.0000,0.2250,0.1550,"247,958.9430"
2,Events,63.1060,59.0000,0.1190,0.1860,"288,150.6380"
5,Referral,67.3160,69.5000,0.2380,0.2060,"278,318.8680"
7,Website,67.4190,61.0000,0.1540,0.1690,"272,491.1860"


C:\Users\ahmad\AppData\Local\Temp\ipykernel_37436\218749098.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Eventual Value By Source

Eventual value matters more than top-of-funnel lead count. This section compares average deal size, total won value, and value share versus lead share.


In [8]:
value_view = source_summary[["source", "lead_share", "won_value_share", "avg_deal_size", "won_value"]].copy()
value_view["value_minus_volume_gap"] = value_view["won_value_share"] - value_view["lead_share"]

display(value_view.sort_values("value_minus_volume_gap", ascending=False).round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
share_plot = value_view.melt(id_vars=["source"], value_vars=["lead_share", "won_value_share"], var_name="share_type", value_name="share")
sns.barplot(data=share_plot, x="source", y="share", hue="share_type", ax=axes[0])
axes[0].set_title("Lead Share vs Won Value Share")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=source_summary.sort_values("avg_deal_size", ascending=False), x="source", y="avg_deal_size", ax=axes[1], color="#2a9d8f")
axes[1].set_title("Average Deal Size By Source")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


,source,lead_share,won_value_share,avg_deal_size,won_value,value_minus_volume_gap
1,Email,0.1440,0.1975,"297,351.8980","5,976,606.0000",0.0535
3,Google Ads,0.1220,0.1605,"283,076.3721","4,856,065.0000",0.0385
0,Cold Call,0.1160,0.1351,"303,676.7805","4,088,799.0000",0.0191
2,Events,0.1180,0.1114,"288,150.6383","3,371,594.0000",-0.0066
5,Referral,0.1260,0.1102,"278,318.8684","3,335,274.0000",-0.0158
4,LinkedIn Ads,0.1020,0.0790,"293,380.0556","2,391,849.0000",-0.0230
7,Website,0.1300,0.1010,"272,491.1860","3,056,468.0000",-0.0290
6,Social Media,0.1420,0.1052,"247,958.9434","3,182,030.0000",-0.0368


C:\Users\ahmad\AppData\Local\Temp\ipykernel_37436\3033586703.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Source Contribution Gaps

This section identifies sources that may be overrated by volume alone and sources that deserve more credit because they over-deliver on conversion or value relative to their lead share.


In [9]:
contribution = source_summary.copy()
contribution["conversion_contribution"] = contribution["conversion_rate"] * contribution["lead_share"]
contribution["win_contribution"] = contribution["win_rate"] * contribution["lead_share"]
contribution["conversion_gap"] = contribution["conversion_contribution"] - contribution["lead_share"]
contribution["value_gap"] = contribution["won_value_share"] - contribution["lead_share"]

contribution["underappreciated_score"] = 100 * (
    0.45 * contribution["value_gap"].rank(pct=True)
    + 0.35 * contribution["conversion_gap"].rank(pct=True)
    + 0.20 * contribution["avg_deal_size"].rank(pct=True)
)
contribution["overrated_volume_score"] = 100 * (
    0.50 * (-contribution["value_gap"]).rank(pct=True)
    + 0.30 * (-contribution["conversion_gap"]).rank(pct=True)
    + 0.20 * contribution["lead_share"].rank(pct=True)
)

display(
    contribution[[
        "source",
        "lead_share",
        "conversion_rate",
        "win_rate",
        "won_value_share",
        "conversion_gap",
        "value_gap",
        "underappreciated_score",
        "overrated_volume_score",
    ]]
    .sort_values("underappreciated_score", ascending=False)
    .round(4)
)

plt.figure(figsize=(13, 8))
sns.scatterplot(
    data=contribution,
    x="lead_share",
    y="won_value_share",
    size="avg_deal_size",
    hue="conversion_rate",
    sizes=(200, 1400),
    palette="plasma",
)
max_share = max(contribution["lead_share"].max(), contribution["won_value_share"].max())
plt.plot([0, max_share], [0, max_share], linestyle="--", color="gray")
for _, row in contribution.iterrows():
    plt.text(row["lead_share"], row["won_value_share"], row["source"], fontsize=10)
plt.title("Lead Share vs Won Value Share By Source")
plt.xlabel("Lead share")
plt.ylabel("Won value share")
plt.show()


,source,lead_share,conversion_rate,win_rate,won_value_share,conversion_gap,value_gap,underappreciated_score,overrated_volume_score
0,Cold Call,0.1160,0.1897,0.2586,0.1351,-0.0940,0.0191,84.3750,31.2500
3,Google Ads,0.1220,0.2131,0.2787,0.1605,-0.0960,0.0385,73.4375,35.6250
1,Email,0.1440,0.1667,0.2917,0.1975,-0.1200,0.0535,66.8750,56.2500
4,LinkedIn Ads,0.1020,0.3725,0.1961,0.0790,-0.0640,-0.0230,66.8750,43.7500
2,Events,0.1180,0.1186,0.1864,0.1114,-0.1040,-0.0066,58.1250,51.2500
5,Referral,0.1260,0.2381,0.2063,0.1102,-0.0960,-0.0158,54.0625,56.8750
7,Website,0.1300,0.1538,0.1692,0.1010,-0.1100,-0.0290,25.0000,85.0000
6,Social Media,0.1420,0.2254,0.1549,0.1052,-0.1100,-0.0368,21.2500,90.0000


C:\Users\ahmad\AppData\Local\Temp\ipykernel_37436\2445065183.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Industry And Source Interaction

A source may look average overall but perform extremely well in specific industries. This cross-section helps avoid cutting useful niche channels.


In [10]:
industry_source = (
    df.groupby(["industry", "source"])
    .agg(
        leads=("lead_id", "count"),
        conversion_rate=("is_converted", "mean"),
        win_rate=("is_closed_won", "mean"),
        avg_deal_size=("max_deal_size", "mean"),
    )
    .reset_index()
)

conv_heatmap = industry_source.pivot(index="industry", columns="source", values="conversion_rate")
value_heatmap = industry_source.pivot(index="industry", columns="source", values="avg_deal_size")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(conv_heatmap, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Conversion Rate By Industry And Source")

sns.heatmap(value_heatmap, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Average Deal Size By Industry And Source")

plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_37436\1433382799.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Statistical Checks

These tests check whether source differences in lead score and sales cycle are likely to be materially separated rather than random noise.


In [11]:
lead_score_groups = [group["lead_score"].dropna().values for _, group in df.groupby("source") if len(group["lead_score"].dropna()) > 1]
cycle_groups = [group["sales_cycle_days"].dropna().values for _, group in df.groupby("source") if len(group["sales_cycle_days"].dropna()) > 1]

if len(lead_score_groups) >= 2:
    score_h, score_p = stats.kruskal(*lead_score_groups)
else:
    score_h, score_p = np.nan, np.nan

if len(cycle_groups) >= 2:
    cycle_h, cycle_p = stats.kruskal(*cycle_groups)
else:
    cycle_h, cycle_p = np.nan, np.nan

rho, rho_p = stats.spearmanr(source_summary["avg_lead_score"], source_summary["win_rate"], nan_policy="omit")

stats_summary = pd.DataFrame(
    {
        "test": [
            "Kruskal-Wallis: lead score by source",
            "Kruskal-Wallis: sales cycle by source",
            "Spearman: avg lead score vs win rate",
        ],
        "statistic": [score_h, cycle_h, rho],
        "p_value": [score_p, cycle_p, rho_p],
    }
)

display(stats_summary.round(4))


,test,statistic,p_value
0,Kruskal-Wallis: lead score by source,12.6280,0.0817
1,Kruskal-Wallis: sales cycle by source,4.6803,0.6989
2,Spearman: avg lead score vs win rate,0.2857,0.4927


## Key Findings

Convert the tables above into a compact source-quality readout for leaders deciding where to spend acquisition budget and how to prioritize sales follow-up.


In [12]:
top_quality = source_summary.sort_values("quality_index", ascending=False).iloc[0]
top_conversion = source_summary.sort_values("conversion_rate", ascending=False).iloc[0]
top_win = source_summary.sort_values("win_rate", ascending=False).iloc[0]
fastest = source_summary.dropna(subset=["avg_sales_cycle_days"]).sort_values("avg_sales_cycle_days").iloc[0]
highest_value = source_summary.sort_values("avg_deal_size", ascending=False).iloc[0]
most_under = contribution.sort_values("underappreciated_score", ascending=False).iloc[0]
most_over = contribution.sort_values("overrated_volume_score", ascending=False).iloc[0]

findings = []
findings.append(f"- Highest overall source quality: {top_quality['source']} with quality index {top_quality['quality_index']:.1f}.")
findings.append(f"- Best lead conversion rate: {top_conversion['source']} at {top_conversion['conversion_rate']:.1%}.")
findings.append(f"- Best win rate: {top_win['source']} at {top_win['win_rate']:.1%}.")
findings.append(f"- Fastest average sales cycle: {fastest['source']} at {fastest['avg_sales_cycle_days']:.1f} days.")
findings.append(f"- Highest average eventual deal size: {highest_value['source']} at {highest_value['avg_deal_size']:.0f}.")
findings.append(f"- Most underappreciated source by value vs volume: {most_under['source']} with value gap {most_under['value_gap']:.1%}.")
findings.append(f"- Most volume-overrated source: {most_over['source']} with value gap {most_over['value_gap']:.1%}.")
if pd.notna(rho):
    findings.append(f"- Average lead score and win rate have Spearman correlation {rho:.3f}, which helps gauge whether sales scoring aligns with realized wins by source.")
if pd.notna(score_p):
    findings.append(f"- Lead score differences by source produce p-value {score_p:.4f}.")
if pd.notna(cycle_p):
    findings.append(f"- Sales cycle differences by source produce p-value {cycle_p:.4f}.")

print("\n".join(findings))


- Highest overall source quality: Cold Call with quality index 78.8.
- Best lead conversion rate: LinkedIn Ads at 37.3%.
- Best win rate: Email at 29.2%.
- Fastest average sales cycle: Google Ads at 56.1 days.
- Highest average eventual deal size: Cold Call at 303677.
- Most underappreciated source by value vs volume: Cold Call with value gap 1.9%.
- Most volume-overrated source: Social Media with value gap -3.7%.
- Average lead score and win rate have Spearman correlation 0.286, which helps gauge whether sales scoring aligns with realized wins by source.
- Lead score differences by source produce p-value 0.0817.
- Sales cycle differences by source produce p-value 0.6989.


## Strategy Implications

Marketing strategy:

- Scale sources that over-deliver on win rate or won value share, even if they are not the largest lead generators.
- Be cautious about channels that dominate raw lead count but underperform on downstream value.
- If a source wins on conversion but not on deal size, position it as a volume engine rather than a premium pipeline engine.

Sales strategy:

- Prioritize faster-cycle sources for shorter-horizon revenue targets.
- Route higher-value sources to stronger account executives if they produce larger eventual deal sizes.
- Use source-by-industry interaction results to specialize outreach and qualification playbooks.

Measurement upgrade:

- Add explicit MQL / SQL timestamps and opportunity-stage history if you want a more precise breakdown of funnel-stage timing by source.
